### 01. Import Depedencies

In [1]:
import warnings
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
                                    StratifiedKFold, 
                                    GridSearchCV
                                    )
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score
warnings.filterwarnings('ignore')

### 02. Load Dataset

In [2]:
X_train = pd.read_csv('artifacts/X_train.csv')
Y_train = pd.read_csv('artifacts/y_train.csv')
X_test = pd.read_csv("artifacts/X_test.csv")
Y_test = pd.read_csv("artifacts/y_test.csv")

### 03. Model Trainig

#### 03.1 define parameters

In [3]:
lr_param_grid = {
    'model__max_iter' : [1000, 5000],
    'model__C': [0.1, 1.0, 10.0]
}

dt_param_grid = {
    'model__max_depth' : [5, 8, 10],
    'model__min_samples_leaf': [2, 5, 10],
    'model__criterion' : ["gini", "entropy"]
}

rf_param_grid = {
    'model__n_estimators' : [100, 200],
    'model__max_depth' : [8, 10, 12],
    'model__min_samples_leaf': [2, 5, 10],
    'model__class_weight': ['balanced', 'balanced_subsample', None]
}

param_grids = {
    'Logistic Regression': lr_param_grid,
    'Decision Tree': dt_param_grid,
    'Random Forest': rf_param_grid
}

### 3.2 Define models

In [4]:
models = {
        'Logistic Regression' : LogisticRegression(),
        'Decision Tree' :DecisionTreeClassifier(),
        'Random Forest' : RandomForestClassifier()
        }

### 3.3 Configure K-Fold CV

In [6]:
cv = StratifiedKFold(
    n_splits = 6,
    random_state = 42,
    shuffle=True 
)

### 3.4 Model Training

In [7]:
grid_search_results={}

In [8]:
for model_name,model in models.items():
    print(f"\n--- Tuning {model_name} ---")

    # Create Pipeline: SMOTE -> Model
    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    param_grid = param_grids[model_name]

    grid_search = GridSearchCV(
                                estimator=pipeline,
                                param_grid=param_grid,
                                cv=cv, scoring='f1',
                                verbose=1, return_train_score=False
                                )
    print(f"Fitting gridSearchCV for {model_name}")

    grid_search.fit(X_train, Y_train)

    grid_search_results[model_name] = grid_search
    
    print(f"{model_name} gridSearchCV completed ...")
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_}")



--- Tuning Logistic Regression ---
Fitting gridSearchCV for Logistic Regression
Fitting 6 folds for each of 6 candidates, totalling 36 fits


  File "c:\Users\www\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\www\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\www\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\www\anaconda3\Lib\subprocess.py", lin

Logistic Regression gridSearchCV completed ...
Best parameters: {'model__C': 0.1, 'model__max_iter': 1000}
Best CV score: 0.6319273436046545

--- Tuning Decision Tree ---
Fitting gridSearchCV for Decision Tree
Fitting 6 folds for each of 18 candidates, totalling 108 fits
Decision Tree gridSearchCV completed ...
Best parameters: {'model__criterion': 'entropy', 'model__max_depth': 5, 'model__min_samples_leaf': 2}
Best CV score: 0.6035969837314419

--- Tuning Random Forest ---
Fitting gridSearchCV for Random Forest
Fitting 6 folds for each of 54 candidates, totalling 324 fits
Random Forest gridSearchCV completed ...
Best parameters: {'model__class_weight': 'balanced_subsample', 'model__max_depth': 8, 'model__min_samples_leaf': 5, 'model__n_estimators': 100}
Best CV score: 0.6303374700527881


In [9]:
grid_search_results

{'Logistic Regression': GridSearchCV(cv=StratifiedKFold(n_splits=6, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                        ('model', LogisticRegression())]),
              param_grid={'model__C': [0.1, 1.0, 10.0],
                          'model__max_iter': [1000, 5000]},
              scoring='f1', verbose=1),
 'Decision Tree': GridSearchCV(cv=StratifiedKFold(n_splits=6, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                        ('model', DecisionTreeClassifier())]),
              param_grid={'model__criterion': ['gini', 'entropy'],
                          'model__max_depth': [5, 8, 10],
                          'model__min_samples_leaf': [2, 5, 10]},
              scoring='f1', verbose=1),
 'Random Forest': GridSearchCV(cv=StratifiedKFold(n_splits=6, random_state=42, shuffle=True),
              estimat

In [11]:
for model_name, grid in grid_search_results.items():
    # Get the best estimator from GridSearchCV
    best_model = grid.best_estimator_
    
    # Make predictions on the test set
    y_pred = best_model.predict(X_test) 

    y_pred_proba = best_model.predict_proba(X_test)[:, 1] 
    
    # Compute metrics
    accuracy = accuracy_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)
    recall = recall_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)
    rf_auc = roc_auc_score(Y_test, y_pred_proba)
    
    print(f"\n--- {model_name} Test Set Metrics ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"ROC-score:  {rf_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(Y_test, y_pred))


--- Logistic Regression Test Set Metrics ---
Accuracy:  0.7321
Precision: 0.4974
Recall:    0.7620
F1-score:  0.6019
ROC-score:  0.8284

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.72      0.80      1033
           1       0.50      0.76      0.60       374

    accuracy                           0.73      1407
   macro avg       0.70      0.74      0.70      1407
weighted avg       0.79      0.73      0.75      1407


--- Decision Tree Test Set Metrics ---
Accuracy:  0.7512
Precision: 0.5250
Recall:    0.6738
F1-score:  0.5902
ROC-score:  0.8048

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.78      0.82      1033
           1       0.53      0.67      0.59       374

    accuracy                           0.75      1407
   macro avg       0.70      0.73      0.71      1407
weighted avg       0.78      0.75      0.76      1407


--- Random Forest Test Set Me